# 🩺 Smart Diabetes Monitoring and Prediction System
## Notebook 2 — Data Preprocessing

Loads raw data from Notebook 1, removes duplicates, caps outliers, splits data, and applies RobustScaler. Saves all preprocessed outputs for Notebook 3.

### Setup — Imports, Config, Settings, Helpers

In [1]:
import os, sys, time, warnings, pickle, json
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, balanced_accuracy_score, log_loss
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from scipy.stats import pointbiserialr

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
    print("✓ imbalanced-learn available")
except ImportError:
    SMOTE_AVAILABLE = False
    print("✗ imbalanced-learn not installed")

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print("✓ XGBoost available")
except ImportError:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
    print("✓ LightGBM available")
except ImportError:
    LGB_AVAILABLE = False

try:
    import catboost as cb
    CAT_AVAILABLE = True
    print("✓ CatBoost available")
except ImportError:
    CAT_AVAILABLE = False

print("\n✅ All libraries loaded.")

✓ imbalanced-learn available
✓ XGBoost available
✓ LightGBM available
✓ CatBoost available

✅ All libraries loaded.


In [ ]:
# ── UPDATE THIS PATH TO YOUR DATASET LOCATION ────────────────────────
DATA_PATH   = r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Dataset\diabetes_binary_health_indicators_BRFSS2015.csv"
OUTPUT_ROOT = Path(r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Outputs_Folder")

DIRS = {
    "models"       : OUTPUT_ROOT / "models",
    "scalers"      : OUTPUT_ROOT / "scalers",
    "plots_eda"    : OUTPUT_ROOT / "plots" / "01_eda",
    "plots_imb"    : OUTPUT_ROOT / "plots" / "02_imbalance",
    "plots_pre"    : OUTPUT_ROOT / "plots" / "03_preprocessing",
    "plots_base"   : OUTPUT_ROOT / "plots" / "04_baseline_models",
    "plots_eng"    : OUTPUT_ROOT / "plots" / "05_feature_engineering",
    "plots_eng_m"  : OUTPUT_ROOT / "plots" / "06_engineered_models",
    "plots_fi"     : OUTPUT_ROOT / "plots" / "07_feature_importance",
    "plots_sel"    : OUTPUT_ROOT / "plots" / "08_selected_models",
    "plots_cmp"    : OUTPUT_ROOT / "plots" / "09_model_comparison",
    "plots_best"   : OUTPUT_ROOT / "plots" / "10_best_model",
    "reports"      : OUTPUT_ROOT / "reports",
    "features"     : OUTPUT_ROOT / "feature_lists",
    "thresholds"   : OUTPUT_ROOT / "thresholds",
    "pipeline"     : OUTPUT_ROOT / "pipeline",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Output root: {OUTPUT_ROOT.resolve()}")

✅ Output root: C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder


In [3]:
PALETTE   = ["#00B4D8","#0077B6","#48CAE4","#90E0EF",
             "#ADE8F4","#023E8A","#03045E","#CAF0F8"]
COLOR_POS = "#EF233C"
COLOR_NEG = "#00B4D8"
COLOR_ACC = "#2EC4B6"
COLOR_AUC = "#FF9F1C"
COLOR_ROC = "#0077B6"
COLOR_PR  = "#2EC4B6"
COLOR_F1  = "#FF9F1C"

sns.set_style("whitegrid")
sns.set_palette(PALETTE)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor"  : "#F8FBFF",
    "axes.edgecolor"  : "#CCCCCC",
    "axes.titlesize"  : 14,
    "axes.labelsize"  : 12,
    "xtick.labelsize" : 10,
    "ytick.labelsize" : 10,
    "font.family"     : "DejaVu Sans",
})

SEED     = 42
CV_FOLDS = 5
TOP_N    = 15
np.random.seed(SEED)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

all_results = {}

print(f"✅ SEED={SEED}  CV_FOLDS={CV_FOLDS}  TOP_N={TOP_N}")

✅ SEED=42  CV_FOLDS=5  TOP_N=15


In [ ]:
def save_pkl(obj, path, label=""):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  [SAVED] {label} → {path}")

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def savefig(fig, path, dpi=150):
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [PLOT]  {path.name}")

def section(title):
    print("\n" + "━"*70)
    print(f"  {title}")
    print("━"*70)

def print_classification_report(y_true, y_pred, model_name, phase=""):
    print(f"\n{'─'*60}")
    print(f"  📊 Classification Report — {model_name}")
    if phase:
        print(f"     Phase: {phase}")
    print(f"{'─'*60}")
    print(classification_report(
        y_true, y_pred,
        target_names=["No Diabetes (0)", "Diabetes (1)"],
        digits=4))
    print(f"{'─'*60}")

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name,
                   cv_obj=None, phase="", verbose=True):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred  = model.predict(X_te)
    y_proba = (model.predict_proba(X_te)[:, 1]
               if hasattr(model, "predict_proba") else None)
    metrics = dict(
        accuracy      = accuracy_score(y_te, y_pred),
        precision     = precision_score(y_te, y_pred, zero_division=0),
        recall        = recall_score(y_te, y_pred, zero_division=0),
        f1            = f1_score(y_te, y_pred, zero_division=0),
        balanced_acc  = balanced_accuracy_score(y_te, y_pred),
        mcc           = matthews_corrcoef(y_te, y_pred),
        roc_auc       = roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
        avg_precision = average_precision_score(y_te, y_proba) if y_proba is not None else np.nan,
        log_loss_val  = log_loss(y_te, y_proba) if y_proba is not None else np.nan,
        train_time_s  = elapsed,
        cv_roc_auc    = np.nan,
    )
    if cv_obj is not None:
        scores = cross_val_score(model, X_tr, y_tr,
                                 cv=cv_obj, scoring="roc_auc", n_jobs=-1)
        metrics["cv_roc_auc"] = scores.mean()
    if verbose:
        print(f"  {name:<35} Acc={metrics['accuracy']:.4f}  "
              f"F1={metrics['f1']:.4f}  Recall={metrics['recall']:.4f}  "
              f"Prec={metrics['precision']:.4f}  AUC={metrics['roc_auc']:.4f}  "
              f"t={elapsed:.1f}s")
        print_classification_report(y_te, y_pred, name, phase)
    return metrics

def plot_roc_pr(y_true, models_proba, title_prefix, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, proba in models_proba.items():
        fpr, tpr, _ = roc_curve(y_true, proba)
        auc = roc_auc_score(y_true, proba)
        axes[0].plot(fpr, tpr, lw=2, label=f"{name}  AUC={auc:.3f}")
        prec, rec, _ = precision_recall_curve(y_true, proba)
        ap = average_precision_score(y_true, proba)
        axes[1].plot(rec, prec, lw=2, label=f"{name}  AP={ap:.3f}")
    axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=.5)
    axes[0].set(xlabel="FPR", ylabel="TPR", title=f"{title_prefix} — ROC Curve")
    axes[0].legend(fontsize=7)
    baseline_ap = y_true.mean()
    axes[1].axhline(baseline_ap, color="gray", linestyle="--",
                    label=f"Baseline AP={baseline_ap:.3f}")
    axes[1].set(xlabel="Recall", ylabel="Precision",
                title=f"{title_prefix} — Precision-Recall Curve")
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, save_dir / f"{title_prefix.replace(' ','_')}_roc_pr.png")
    plt.show()

def results_to_df(results_dict):
    rows = []
    for phase, models in results_dict.items():
        for mname, m in models.items():
            row = {"Phase": phase, "Model": mname}
            row.update(m)
            rows.append(row)
    return pd.DataFrame(rows)

def get_models():
    m = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=SEED, class_weight="balanced"),
        "Random Forest"      : RandomForestClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Extra Trees"        : ExtraTreesClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Gradient Boosting"  : GradientBoostingClassifier(
            n_estimators=200, random_state=SEED),
        "AdaBoost"           : AdaBoostClassifier(
            n_estimators=100, random_state=SEED),
    }
    if XGB_AVAILABLE:
        m["XGBoost"] = xgb.XGBClassifier(
            n_estimators=200, use_label_encoder=False,
            eval_metric="logloss", random_state=SEED, n_jobs=-1,
            scale_pos_weight=6)
    if LGB_AVAILABLE:
        m["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=200, random_state=SEED, n_jobs=-1,
            class_weight="balanced", verbose=-1,
            scale_pos_weight=6,
            min_child_samples=5,
            min_split_gain=0.0)
    if CAT_AVAILABLE:
        m["CatBoost"] = cb.CatBoostClassifier(
            iterations=200, random_state=SEED, verbose=0,
            auto_class_weights="Balanced")
    return m

print("✅ All helper functions defined.")

✅ All helper functions defined.


### Load Outputs from Notebook 1

In [5]:
df_raw           = load_pkl(DIRS["reports"] / "df_raw.pkl")
TARGET           = "Diabetes_binary"
FEATURES_ORIGINAL = load_pkl(DIRS["reports"] / "features_original.pkl")
binary_cols      = load_pkl(DIRS["reports"] / "binary_cols.pkl")
cont_cols        = load_pkl(DIRS["reports"] / "cont_cols.pkl")

print(f"  df_raw shape      : {df_raw.shape}")
print(f"  Original features : {len(FEATURES_ORIGINAL)}")
print(f"  Binary cols       : {len(binary_cols)}")
print(f"  Continuous cols   : {len(cont_cols)}")
print("✅ Notebook 1 outputs loaded.")

  df_raw shape      : (253680, 22)
  Original features : 21
  Binary cols       : 14
  Continuous cols   : 7
✅ Notebook 1 outputs loaded.


### Step 7 — Data Preprocessing

In [6]:
section("STEP 7 — DATA PREPROCESSING")

df = df_raw.copy()

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"  Duplicates removed : {before - len(df)}  → {len(df):,} rows")

# Outlier capping (IQR 1st-99th percentile on continuous cols)
outlier_log = {}
for col in cont_cols:
    Q1, Q3 = df[col].quantile(0.01), df[col].quantile(0.99)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    df[col] = df[col].clip(lower=lo, upper=hi)
    outlier_log[col] = {"lower": float(lo), "upper": float(hi), "capped": int(n_out)}
    print(f"  Clipped [{col:<12}]: {n_out:>5} values → [{lo:.2f}, {hi:.2f}]")

save_pkl(outlier_log, DIRS["scalers"] / "outlier_caps.pkl", "Outlier caps")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 7 — DATA PREPROCESSING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Duplicates removed : 24206  → 229,474 rows
  Clipped [BMI         ]:     0 values → [-30.00, 98.00]
  Clipped [GenHlth     ]:     0 values → [-5.00, 11.00]
  Clipped [MentHlth    ]:     0 values → [-45.00, 75.00]
  Clipped [PhysHlth    ]:     0 values → [-45.00, 75.00]
  Clipped [Age         ]:     0 values → [-17.00, 31.00]
  Clipped [Education   ]:     0 values → [-4.00, 12.00]
  Clipped [Income      ]:     0 values → [-9.50, 18.50]
  [SAVED] Outlier caps → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\scalers\outlier_caps.pkl


In [7]:
# Stratified 80/20 Split
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)

print(f"  Train : {len(X_train_raw):>7,}  | Class balance: {y_train.value_counts(normalize=True).round(4).to_dict()}")
print(f"  Test  : {len(X_test_raw):>7,}  | Class balance: {y_test.value_counts(normalize=True).round(4).to_dict()}")

# RobustScaler
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=X_train_raw.columns, index=X_train_raw.index)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=X_test_raw.columns, index=X_test_raw.index)

save_pkl(scaler,         DIRS["scalers"] / "robust_scaler.pkl",  "RobustScaler")
save_pkl(X_train_scaled, DIRS["reports"] / "X_train_scaled.pkl", "X_train_scaled")
save_pkl(X_test_scaled,  DIRS["reports"] / "X_test_scaled.pkl",  "X_test_scaled")
save_pkl(y_train,        DIRS["reports"] / "y_train.pkl",        "y_train")
save_pkl(y_test,         DIRS["reports"] / "y_test.pkl",         "y_test")
save_pkl(df,             DIRS["reports"] / "df_preprocessed.pkl","df_preprocessed")

print("\n✅ Preprocessing complete. All outputs saved.")

  Train : 183,579  | Class balance: {0: 0.8471, 1: 0.1529}
  Test  :  45,895  | Class balance: {0: 0.8471, 1: 0.1529}
  [SAVED] RobustScaler → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\scalers\robust_scaler.pkl
  [SAVED] X_train_scaled → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\X_train_scaled.pkl
  [SAVED] X_test_scaled → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\X_test_scaled.pkl
  [SAVED] y_train → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\y_train.pkl
  [SAVED] y_test → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\y_test.pkl
  [SAVED] df_preprocessed → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\df_preprocessed.pkl

✅ Preprocessing complete. All outputs saved.


In [8]:
# Visualise scaled distributions
fig, axes = plt.subplots(1, len(cont_cols), figsize=(len(cont_cols) * 3.5, 4))
if len(cont_cols) == 1: axes = [axes]
for i, col in enumerate(cont_cols):
    axes[i].hist(X_train_scaled[col], bins=30, color=COLOR_NEG,
                 alpha=0.7, edgecolor="white", density=True)
    axes[i].set_title(col, fontweight="bold", fontsize=9)
fig.suptitle("Scaled Feature Distributions (Train Set)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_pre"] / "01_scaled_distributions.png")
plt.show()

print("\n  Train shape :", X_train_scaled.shape)
print("  Test  shape :", X_test_scaled.shape)

  [PLOT]  01_scaled_distributions.png

  Train shape : (183579, 21)
  Test  shape : (45895, 21)


### ✅ Notebook 2 Complete
All preprocessed data saved. Run Notebook 3 next.